# ACF Comparison
Computes and overlays ACF curves for sfGFP, GFPuv, GFP_ex (end), GFP_sx (start) on a single plot.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
current_dir = Path().resolve()
from microlive.imports import *
from microlive import microscopy as mi
from pipeline_time_courses import compute_autocorrelation_for_dataset, extract_intensity_distributions, plot_intensity_distributions
from scipy.stats import gaussian_kde
import matplotlib.patheffects as pe


import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# ── Dataset Paths ─────────────────────────────────────────────────────────────
data_folder_sf        = Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/sfGFP/results')
#data_folder_uv        = Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/GFPuv/results')
data_folder_end_xbp1  = Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/pRS038/results')
data_folder_start_xbp1= Path('/Volumes/Luis_DRIVE/CoF Manuscript LIFs/CoF AC/pRS048/results')

list_datasets = [data_folder_sf, data_folder_end_xbp1, data_folder_start_xbp1] # data_folder_uv
list_names    = ['sfGFP',  'sfGFP_ex', 'sfGFP_sx'] # 'GFPuv',

# ── Colour palette (one per dataset) ─────────────────────────────────────────
#list_colors = ['#2196F3', '#FF5722', '#9C27B0']  # blue, green, orange, purple #  '#4CAF50',
list_colors = ['gray', 'tab:orange', 'tab:blue']

In [ ]:
# ── Shared ACF Parameters ─────────────────────────────────────────────────────
step_size_in_sec                  = 5
start_lag                         = 1
channel_index                     = 1          # channel 1 for all datasets
selected_field                    = 'spot_int_ch_'
min_percentage_data_in_trajectory = 0.25
max_missing_frames                = 1
downsample                        = False
downsampling_factor               = 3
control_spots_mode                = False
use_global_mean                   = False
MAD_THRESHOLD_FACTOR              = 6
multi_tau_raw_points              = 60
multi_tau_bins_per_stage          = 16
min_snr                           = 0.5
smooth_window                     = 1
remove_outliers                   = True
correct_baseline                  = True
multi_tau                         = True
max_lag                           = 200
x_axes_min_max_list_values        = [-10, 1000]
y_axes_min_max_list_values        = [-0.02, 0.04]
fit_type                          = 'exponential'
de_correlation_threshold          = 0.001
use_linear_projection_for_lag_0   = True
gene_length                       = 1826       # codons (Xbp1u)

In [ ]:
# ── Run ACF for each dataset (suppress individual plots while running) ────────
all_results = []

for data_folder, name in zip(list_datasets, list_names):
    results_folder = data_folder / 'processing'
    results_folder.mkdir(exist_ok=True)
    #plot_name = f'{name}_ACF.svg'
    print(f'Running ACF for {name} ...')
    r = compute_autocorrelation_for_dataset(
        dataset                          = 'cof',
        data_folder                      = data_folder,
        results_folder                   = results_folder,
        selected_field                   = selected_field,
        channel_index                    = channel_index,
        step_size_in_sec                 = step_size_in_sec,
        start_lag                        = start_lag,
        min_percentage_data_in_trajectory= min_percentage_data_in_trajectory,
        max_missing_frames               = max_missing_frames,
        downsample                       = downsample,
        downsampling_factor              = downsampling_factor,
        use_global_mean                  = use_global_mean,
        control_spots_mode               = control_spots_mode,
        correct_baseline                 = correct_baseline,
        min_snr                          = min_snr,
        smooth_window                    = smooth_window,
        remove_outliers                  = remove_outliers,
        MAD_THRESHOLD_FACTOR             = MAD_THRESHOLD_FACTOR,
        multi_tau                        = multi_tau,
        multi_tau_raw_points             = multi_tau_raw_points,
        multi_tau_bins_per_stage         = multi_tau_bins_per_stage,
        x_axes_min_max_list_values       = x_axes_min_max_list_values,
        max_lag                          = max_lag,
        index_max_lag_for_fit            = None,
        fit_type                         = fit_type,
        de_correlation_threshold         = de_correlation_threshold,
        use_linear_projection_for_lag_0  = use_linear_projection_for_lag_0,
        verbose                          = False,
        simulation_mode                  = False,
        SSA_data                         = None,
        line_color                       = (0.5, 0.5, 0.5),
        line_color_fit                   = 'dimgray',
        plot_name                        = None,
        save_plots                       = False,
        figsize                          = (3.2, 2.2),
    )
    all_results.append(r)
    dwell_time = r['dwell_time']
    ke = gene_length / dwell_time
    ki = 1.0 / (r['mean_correlation'][1] * dwell_time)
    print(f'  -> dwell_time={dwell_time:.1f}s  ke={ke:.2f} cod/s  ki={ki:.4f} rib/s')

print('\nAll datasets done.')

In [ ]:
# ── Kinetics Summary Table ────────────────────────────────────────────────────
print(f"{'Dataset':<12}  {'dwell(s)':<10}  {'ki (rib/s)':<12}  {'ke (cod/s)':<12}")
print('-' * 52)
for r, name in zip(all_results, list_names):
    dwell_time = r['dwell_time']
    ke = gene_length / dwell_time
    ki = 1.0 / (r['mean_correlation'][1] * dwell_time)
    print(f"{name:<12}  {dwell_time:<10.1f}  {ki:<12.4f}  {ke:<12.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5.5))
for r, name, color in zip(all_results, list_names, list_colors):
    lags        = np.array(r['lags'])
    mean_corr   = np.array(r['mean_correlation'])
    std_corr    = np.array(r['std_correlation'])
    dwell_time  = r['dwell_time']
    n_cells     = r['number_of_cells_final']   # post-filter: cells with ≥1 surviving trajectory
    n_traces    = r['number_of_trajectories_final']  # post-filter: trajectories after MAD removal
    x_min, x_max = x_axes_min_max_list_values
    mask = (lags > 0) & (lags <= x_max)
    ax.plot(lags[mask], mean_corr[mask],
            color=color, linewidth=1.5,
            label=f"{name}  |  {n_cells} cells | {n_traces} traces")
    ax.fill_between(lags[mask],
                    mean_corr[mask] - std_corr[mask],
                    mean_corr[mask] + std_corr[mask],
                    color=color, alpha=0.15)
# Decorations
ax.axhline(0, color='black', linewidth=0.6, linestyle='--', alpha=0.5)
ax.set_xlabel('τ (s)', fontsize=16)
ax.set_ylabel('G(τ)', fontsize=16)
# tick size
ax.tick_params(axis='both', which='major', labelsize=16)
#ax.legend(fontsize=12, framealpha=0.9)
ax.legend(fontsize=12, framealpha=0.9,
          loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=1)
ax.set_xlim(x_axes_min_max_list_values)
ax.set_xlim(0, 800)
ax.set_ylim(-0.02, 0.07)
ax.grid(False)
plt.tight_layout()
plt.savefig('ACF_comparison_ch1.svg', dpi=300, bbox_inches='tight')
plt.savefig('ACF_comparison_ch1.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ACF_comparison_ch1.svg / .png')

In [ ]:
print(f"{'Dataset':<12}  {'dwell(s)':<10}  {'ki (rib/s)':<12}  {'ke (cod/s)':<12}  {'cells':<7}  {'traces':<7}")
print('-' * 68)
for r, name in zip(all_results, list_names):
    dwell_time = r['dwell_time']
    ke = gene_length / dwell_time
    ki = 1.0 / (r['mean_correlation'][1] * dwell_time)
    n_cells  = r['number_of_cells_final']
    n_traces = r['number_of_trajectories_final']
    print(f"{name:<12}  {dwell_time:<10.1f}  {ki:<12.4f}  {ke:<12.2f}  {n_cells:<7}  {n_traces:<7}")

In [ ]:
selected_field_dist = selected_field   # column prefix
display_mode        = 1               # 1 = mean per particle | 2 = all timepoints | 3 = snapshot
x_label             = 'Nascent Chain Spot Intensity' #'Spot Intensity (a.u.)'
xlim                = (-10, 1500)           # e.g. (0, 5000) or None
int_dist_results = []
for data_folder, name in zip(list_datasets, list_names):
    print(f'  {name} ...', end=' ')
    dist = extract_intensity_distributions(
        data_folder    = data_folder,
        dataset        = 'cof',
        selected_field = selected_field_dist,
        channel_index  = 1,
        min_snr        = min_snr,
        timepoint_frame= None,
        verbose        = False,
    )
    int_dist_results.append(dist)
    print(f'{dist["n_particles"]} particles')
plot_intensity_distributions(
    dist_results   = int_dist_results,
    list_names     = list_names,
    list_colors    = list_colors,
    channel_index  = 1,
    mode           = display_mode,
    x_label        = x_label,
    xlim           = xlim,
    figsize = (5, 5),
    save_name      = f'intensity_dist_ch',
)

In [ ]:
selected_field_dist = selected_field   # column prefix
display_mode        = 1               # 1 = mean per particle | 2 = all timepoints | 3 = snapshot
x_label             = 'Folding Spot Intensity'
xlim                = (-10, 3500)           # e.g. (0, 5000) or None
int_dist_results = []
for data_folder, name in zip(list_datasets, list_names):
    print(f'  {name} ...', end=' ')
    dist = extract_intensity_distributions(
        data_folder    = data_folder,
        dataset        = 'cof',
        selected_field = selected_field_dist,
        channel_index  = 0,
        min_snr        = min_snr,
        timepoint_frame= None,
        verbose        = False,
    )
    int_dist_results.append(dist)
    print(f'{dist["n_particles"]} particles')
plot_intensity_distributions(
    dist_results   = int_dist_results,
    list_names     = list_names,
    list_colors    = list_colors,
    channel_index  = 0,
    mode           = display_mode,
    x_label        = x_label,
    xlim           = xlim,
    figsize = (5, 5),
    save_name      = f'intensity_dist_ch',
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for data_folder, name, color in zip(list_datasets, list_names, list_colors):
    d_amp = extract_intensity_distributions(
        data_folder=data_folder, dataset='cof',
        selected_field='psf_amplitude_ch_', channel_index=channel_index,
        min_snr=min_snr, verbose=False,
    )
    d_sig = extract_intensity_distributions(
        data_folder=data_folder, dataset='cof',
        selected_field='psf_sigma_ch_', channel_index=channel_index,
        min_snr=min_snr, verbose=False,
    )

    amp = d_amp['mean_per_particle']
    sig = d_sig['mean_per_particle']
    n   = min(len(amp), len(sig))
    amp, sig = amp[:n], sig[:n]

    # ── Clip to plot limits to avoid KDE smearing outside the axes ──
    mask = (sig >= 1.4) & (sig <= 2.0) & (amp >= 100) & (amp <= 7000)
    sig_c, amp_c = sig[mask], amp[mask]

    try:
        xy  = np.vstack([sig_c, amp_c])
        kde = gaussian_kde(xy, bw_method=0.25)          # tighter BW = less smear

        xg = np.linspace(1.4, 2.0,  120)
        yg = np.linspace(100, 7000, 120)
        XX, YY = np.meshgrid(xg, yg)
        ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)

        # Normalise so every dataset has the same peak density = 1
        ZZ /= ZZ.max()

        # Filled contours at 20 / 50 / 80 % of peak density
        cf = ax.contourf(XX, YY, ZZ,
                         levels=[0.20, 0.50, 0.80, 1.01],
                         colors=[color], alpha=0.18)   # light fill

        # Hard outline at 20 % (outermost boundary)
        ax.contour(XX, YY, ZZ,
                   levels=[0.20, 0.50, 0.80],
                   colors=[color], linewidths=[0.7, 1.0, 1.4], alpha=0.85)

        # Median crosshair  ──  most robust location estimate
        med_x, med_y = np.median(sig_c), np.median(amp_c)
        ax.plot(med_x, med_y, marker='+', color=color,
                markersize=10, markeredgewidth=1.8,
                path_effects=[pe.withStroke(linewidth=3, foreground='white')],
                zorder=5, label=f'{name}  (n={n})')

    except Exception:
        # Fallback: tiny scatter if KDE fails (too few points)
        ax.scatter(sig_c, amp_c, color=color, alpha=0.4, s=8,
                   linewidths=0, label=f'{name}  (n={n})')

ax.set_xlabel('PSF σ (px)', fontsize=11)
ax.set_ylabel('PSF Amplitude (a.u.)', fontsize=11)
ax.set_title(f'PSF Amplitude vs Sigma — Channel {channel_index}',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.2, linewidth=0.4)
ax.set_xlim(1.5, 1.9)
ax.set_ylim(500, 4100)
plt.tight_layout()
plt.savefig(f'psf_amplitude_vs_sigma_ch{channel_index}.svg', dpi=300, bbox_inches='tight')
plt.savefig(f'psf_amplitude_vs_sigma_ch{channel_index}.png', dpi=300, bbox_inches='tight')
plt.show()
